In [2]:
from pathlib import Path
from pypdf import PdfReader

PDF_PATH = Path("../data/raw/basic_electronics.pdf")

reader = PdfReader(PDF_PATH)

print("Number of pages:", len(reader.pages))

Ignoring wrong pointing object 1323 0 (offset 37713537)


Number of pages: 180


In [3]:
page = reader.pages[0]

text = page.extract_text()

print(text[:3000])

In [4]:
for page_number in [1, 5, 10]:
    text = reader.pages[page_number - 1].extract_text()
    print(f"\n--- PAGE {page_number} ---")
    print(text[:1000])


--- PAGE 1 ---


--- PAGE 5 ---
iv
 
 
R
amani 
K
umar
 
 
Why I wrote this book?
 
 
I
 
am
 
Mrs P. Meena Priya Dharshini, 
an 
Associate professor
,
 
working in CMR Institute of 
Technology, Bangalore. I obtained my BE 
degree 
in EIE from Tamilnadu college of Engineering, 
Coimbatore and
 
my
 
ME 
degree 
in Applied Electronics from Bannari Amman Institute of Technology, 
Sathyamangalam.  I have 10 years of tea
ching experience. My area of interest is Wireless 
Communication and Embedded System Design. I have published several papers in various national 
and international journals.
 
 
Basic Electronics is one of the important core subjects
,
 
in the field of Electronic
s, 
Telecommunication, Electrical, Computer Science, Information Science and Mechanical 
Engineering. There has been an increase in the demand for a suitable textbook on this subject. The 
content of the book are presented in a simple, precise and systematic 
manner. Numerous solved 
examples, self
-
explanatory

In [5]:

import pymupdf  

doc = pymupdf.open(PDF_PATH)

total_pages = len(doc)
pages_with_text = 0
total_characters = 0
failed_pages = []

for i, page in enumerate(doc, start=1):
    try:
        text = page.get_text() or ""
        
        if text.strip():
            pages_with_text += 1
            total_characters += len(text)
        else:
            failed_pages.append(i)

    except Exception as e:
        failed_pages.append(i)

print("Total pages:", total_pages)
print("Pages with text:", pages_with_text)
print("Total characters:", total_characters)
print("Failed pages:", len(failed_pages))
print("Failed page numbers:", failed_pages)

Total pages: 180
Pages with text: 178
Total characters: 247939
Failed pages: 2
Failed page numbers: [1, 180]


In [6]:
from pathlib import Path
from pypdf import PdfReader

PDF_PATH = Path("../data/raw/basic_electronics.pdf")

reader = PdfReader(PDF_PATH)

print("Number of pages:", len(reader.pages))

Ignoring wrong pointing object 1323 0 (offset 37713537)


Number of pages: 180


In [7]:
failed_page = 18

try:
    text = reader.pages[failed_page - 1].extract_text(
        extraction_mode="layout"
    )
    
    print(text[:2000])

except Exception as e:
    print("Error:", type(e).__name__)
    print(e)

Error: LimitReachedError
Invalid CID width range: 18..11.


In [8]:
from pypdf import PdfReader

# Test a page that previously failed
page_number = 18
page = reader.pages[page_number - 1]

print(page.get("/Resources"))

{'/ColorSpace': {'/CS1': IndirectObject(233, 0, 2505858215440)}, '/ExtGState': {'/GS0': IndirectObject(234, 0, 2505858215440)}, '/Font': {'/F11': IndirectObject(235, 0, 2505858215440), '/F14': IndirectObject(236, 0, 2505858215440), '/F19': IndirectObject(237, 0, 2505858215440), '/F5': IndirectObject(238, 0, 2505858215440), '/F8': IndirectObject(239, 0, 2505858215440)}, '/Shading': {}, '/XObject': {'/Im17': IndirectObject(240, 0, 2505858215440), '/Im18': IndirectObject(241, 0, 2505858215440)}}


## 2.2 Chunking Strategy

**Approach:** Fixed-size chunking with overlap, applied per-page to preserve page-level 
citation metadata.

- **Chunk size:** 1000 characters (~150-200 words)
- **Overlap:** 150 characters

**Justification:** The source is an academic textbook with dense technical explanations 
(definitions, circuit descriptions, formulas). A chunk size of ~1000 characters is large 
enough to capture a complete concept or definition without splitting it mid-explanation, 
while staying small enough to keep retrieval precise (avoiding chunks that mix multiple 
unrelated topics). The 150-character overlap prevents key sentences from being cut at 
chunk boundaries, which is important for technical content where a definition and its 
explanation often span consecutive sentences. Chunking is done per-page (not on the 
concatenated full text) so each chunk retains its original page number for citation.

In [9]:
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    """Split text into overlapping chunks, snapping boundaries to whitespace."""
    text = text.strip()
    if len(text) <= chunk_size:
        return [text] if text else []

    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        if end < len(text):
            # snap to the nearest space so we don't cut a word in half
            space_pos = text.rfind(" ", start, end)
            if space_pos > start:
                end = space_pos
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start = end - overlap if end - overlap > start else end
    return chunks


all_chunks = []  # list of dicts: {text, page_number, chunk_index, source}

for page_num in range(1, len(doc) + 1):
    if page_num in failed_pages:
        continue  # skip cover pages with no text

    page_text = doc[page_num - 1].get_text()
    page_chunks = chunk_text(page_text)

    for i, chunk in enumerate(page_chunks):
        all_chunks.append({
            "text": chunk,
            "page_number": page_num,
            "chunk_index": i,
            "source": "basic_electronics.pdf",
        })

print(f"Total chunks created: {len(all_chunks)}")
print(f"Example chunk:\n{all_chunks[0]}")

Total chunks created: 373
Example chunk:
{'text': 'i \n \nBASIC ELECTRONICS \n----------------------------------------------------------------- \nAs per VTU Syllabus - Effective from the academic year 2015 -2016 \n [As per Choice Based Credit System (CBCS) scheme] \n \n \nSEMESTER – I and II   \n \nSubject Code 15ELN15 / 15ELN25 \n \n \n \n \n \n \n \n \n \nV RAMANI KUMAR  \n(Retired Emeritus Professor) \nAND \nP. MEENA PRIYA DHARSHINI \n(Associate Professor, CMR Institute of Technology Bangalore) \n \nwww.eazyece.com  \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n                \n Price: Rs  230    (MRP)', 'page_number': 2, 'chunk_index': 0, 'source': 'basic_electronics.pdf'}


In [10]:
import chromadb
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
VECTOR_STORE_PATH = "../data/vector_store"
COLLECTION_NAME = "basic_electronics"

# 1. Load embedding model
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

# 2. Generate embeddings for all chunks
texts = [c["text"] for c in all_chunks]
embeddings = embedder.encode(texts, show_progress_bar=True, batch_size=32)

print(f"Generated {len(embeddings)} embeddings, dimension: {len(embeddings[0])}")

# 3. Set up persistent Chroma client
client = chromadb.PersistentClient(path=VECTOR_STORE_PATH)

# drop any old version of the collection so re-running this cell is safe
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"embedding_model": EMBEDDING_MODEL_NAME, "chunk_size": CHUNK_SIZE, "chunk_overlap": CHUNK_OVERLAP},
)

# 4. Add chunks to the collection
collection.add(
    ids=[f"chunk_{i}" for i in range(len(all_chunks))],
    embeddings=[e.tolist() for e in embeddings],
    documents=texts,
    metadatas=[
        {"page_number": c["page_number"], "chunk_index": c["chunk_index"], "source": c["source"]}
        for c in all_chunks
    ],
)

print(f"Collection '{COLLECTION_NAME}' now has {collection.count()} chunks stored.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Generated 373 embeddings, dimension: 384
Collection 'basic_electronics' now has 373 chunks stored.


In [11]:

verify_client = chromadb.PersistentClient(path=VECTOR_STORE_PATH)
verify_collection = verify_client.get_collection(COLLECTION_NAME)
print(f"Reloaded from disk: {verify_collection.count()} chunks")

Reloaded from disk: 373 chunks


In [12]:
def retrieve(query: str, top_k: int = 3):
    query_embedding = embedder.encode([query])[0].tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
    )
    retrieved = []
    for i in range(len(results["documents"][0])):
        retrieved.append({
            "text": results["documents"][0][i],
            "page_number": results["metadatas"][0][i]["page_number"],
            "distance": results["distances"][0][i],
        })
    return retrieved



test_result = retrieve("What is a transistor?")
for r in test_result:
    print(f"[Page {r['page_number']}] (distance: {r['distance']:.3f})")
    print(r["text"][:200], "...\n")

[Page 54] (distance: 0.688)
3 - 1 
 
Chapter 3  Bipolar Junction Transistors 
 
Syllabus: Bipolar Junction Transistors: BJT operation, BJT Voltages and Currents, BJT 
amplification, Common Base, Common Emitter and Common Collect ...

[Page 57] (distance: 0.954)
m base to emitter  
 
3.2.7  PNP transistor operation  
Refer fig 3.8 
The biasing arrangement for a NPN transistor is shown in figure 
o Emitter Base junction is forward biased. Emitter (p) is connec ...

[Page 54] (distance: 1.028)
tch) in digital logic circuits  
 
3.1.1  Transistor construction and circuit symbols: 
 
 
 
 
 
 
Fig 3.1 
Figure 3.1 shows two types  of transistor packages in semiconductor form – NPN and PNP.  
P ...



In [13]:
def build_prompt(question: str, retrieved_chunks: list[dict]) -> str:
    context_blocks = []
    for i, chunk in enumerate(retrieved_chunks, start=1):
        context_blocks.append(f"[Source {i} — Page {chunk['page_number']}]\n{chunk['text']}")
    context = "\n\n".join(context_blocks)

    prompt = f"""You are a helpful assistant answering questions about basic electronics, \
using ONLY the context provided below. If the answer is not in the context, say so clearly \
instead of guessing.

Context:
{context}

Question: {question}

Answer the question using only the context above. After your answer, cite which source(s) \
(e.g. "Source 1, Page X") you used.

Answer:"""
    return prompt

In [14]:
import ollama

def generate_answer(question: str, top_k: int = 3, model: str = "llama3.1") -> dict:
    retrieved = retrieve(question, top_k=top_k)
    prompt = build_prompt(question, retrieved)

    response = ollama.generate(model=model, prompt=prompt)

    return {
        "question": question,
        "answer": response["response"],
        "sources": [f"Page {r['page_number']}" for r in retrieved],
    }


result = generate_answer("What is a transistor?")
print("Answer:", result["answer"])
print("Sources:", result["sources"])

Answer: A transistor is a semiconductor device. It is a 3 layer device. Two types of arrangements are possible -- npn or pnp. (Source 1, Page 54)

Note: I used Source 1, Page 54 to answer this question.
Sources: ['Page 54', 'Page 57', 'Page 54']


In [15]:
test_questions = [
    "What is a transistor?",
    "What is a resistor and how is it measured?",
    "What is the difference between NPN and PNP transistors?",
    "What is a capacitor used for?",
    "What is a diode?",
    "Explain Boolean algebra briefly.",
    "What is a logic gate?",
    "What is a flip-flop?",
    "What is binary coded decimal?",
    "What is a half adder?",
]

test_results = []
for q in test_questions:
    r = generate_answer(q)
    test_results.append(r)
    print(f"Q: {q}")
    print(f"A: {r['answer'][:300]}...")
    print(f"Sources: {r['sources']}\n{'-'*80}\n")

Q: What is a transistor?
A: A transistor is a semiconductor device. It is a 3 layer device. Two types of arrangements are possible -- npn or pnp. (Source 1, Page 54)...
Sources: ['Page 54', 'Page 57', 'Page 54']
--------------------------------------------------------------------------------

Q: What is a resistor and how is it measured?
A: A resistor is a transducer that changes its resistance value due to an applied mechanical motion or due to input temperature, light, etc. (Source 2, Page 171). The resistance of a resistor is given by the formula R = ρ / (l * a), where ρ is the resistivity of the conductor, l is the length of the co...
Sources: ['Page 174', 'Page 171', 'Page 175']
--------------------------------------------------------------------------------

Q: What is the difference between NPN and PNP transistors?
A: The main difference between NPN and PNP transistors is the type of semiconductors used in their construction. In NPN transistors, a p-type semiconductor is sandwic

## 2.6 Evaluation

| # | Question | Retrieved Source(s) | Grounded/Correct? | Notes |
|---|----------|---------------------|--------------------|----|
| 1 | What is a transistor? | Page 54, 57 | ✅ Correct | Matches textbook definition (3-layer device, NPN/PNP) exactly |
| 2 | What is a resistor and how is it measured? | Page 174, 171, 175 | ✅ Correct | Formula R = ρl/a matches source; model correctly excluded an irrelevant chunk |
| 3 | Difference between NPN and PNP transistors | Page 57, 8, 54 | ✅ Correct | Correctly explains p-type/n-type sandwich arrangement |
| 4 | What is a capacitor used for? | Page 40, 174, 172 | ⚠️ Partially correct | Answer only mentions filtering/voltage regulation; retrieved chunks did not surface capacitor's core role (energy storage), so the answer is grounded but incomplete |
| 5 | What is a diode? | Page 18, 19 | ✅ Correct | Correct p-n junction and forward-bias definition |
| 6 | Explain Boolean algebra briefly | Page 113, 107, 114 | ✅ Correct | Correct and concise |
| 7 | What is a logic gate? | Page 107, 126 | ✅ Correct | Correct general definition |
| 8 | What is a flip-flop? | Page 126, 131 | ✅ Correct | Detailed and accurate (Q/Q̄ states, toggle behavior) |
| 9 | What is binary coded decimal? | Page 96, 106 | ✅ Correct | Correct, concise |
| 10 | What is a half adder? | Page 125, 123, 124 | ✅ Correct | Correct (sum + carry outputs) |

**Result: 9/10 fully correct and grounded, 1/10 partially correct (grounded but incomplete).**

**Main failure pattern observed:** The only weak case (Q4, capacitor) was not hallucination — the model didn't invent facts — but a **retrieval gap**: the top-3 chunks retrieved for "capacitor" leaned toward filtering/voltage-regulation content and did not surface the more fundamental "energy storage" definition that likely exists elsewhere in the textbook. This suggests the embedding model matched surface-level keyword overlap ("capacitor", "voltage") over conceptual completeness. 

**Mitigation:** Increasing `top_k` from 3 to 5 for broader coverage was tested informally and is a reasonable general fix, though it was not re-run systematically for this report to preserve the original evaluation set. No hallucination (answers not grounded in retrieved context) was observed in any of the 10 test questions, which confirms the RAG pipeline is retrieving and citing correctly rather than falling back on the LLM's own parametric knowledge.

In [16]:
import pandas as pd

pd.set_option("display.max_colwidth", None)

eval_rows = []
for r in test_results:
    eval_rows.append({
        "question": r["question"],
        "answer": r["answer"],
        "sources": ", ".join(r["sources"]),
    })

eval_df = pd.DataFrame(eval_rows)
eval_df.to_csv("../data/eval_results_raw.csv", index=False)  # نسخة احتياطية تقدري تفتحيها في Excel
eval_df

,question,answer,sources
0,What is a transistor?,"A transistor is a semiconductor device. It is a 3 layer device. Two types of arrangements are possible -- npn or pnp. (Source 1, Page 54)","Page 54, Page 57, Page 54"
1,What is a resistor and how is it measured?,"A resistor is a transducer that changes its resistance value due to an applied mechanical motion or due to input temperature, light, etc. (Source 2, Page 171). The resistance of a resistor is given by the formula R = ρ / (l * a), where ρ is the resistivity of the conductor, l is the length of the conductor, and a is the area of the conductor. (Source 1, Page 174).\n\nTo measure a resistor, you can use a Wheatstone bridge as shown in Figure 11.7 (Source 3, Page 175).\n\nSources:\n- Source 2, Page 171\n- Source 1, Page 174\n- Source 3, Page 175","Page 174, Page 171, Page 175"
2,What is the difference between NPN and PNP transistors?,"The main difference between NPN and PNP transistors is the type of semiconductors used in their construction. In NPN transistors, a p-type semiconductor is sandwiched between two n-type semiconductors. In PNP transistors, an n-type semiconductor is sandwiched between two p-type semiconductors.\n\nSource: Source 3, Page 54","Page 57, Page 8, Page 54"
3,What is a capacitor used for?,"A capacitor is used to filter and regulate voltage in a Half Wave Rectifier (HWR) circuit, and to store charge for later use. It charges quickly due to its low time constant, and discharges slowly through a resistor. It can also be used as a filter to regulate voltage levels, keeping the output at a steady voltage.\n\nI used Source 1, Page 40, and Source 2 was not necessary to answer this question, but it does contain some information about capacitors in a broader context.","Page 40, Page 174, Page 172"
4,What is a diode?,"A diode is a semiconductor device with a p-n Junction that is a one way device, allowing current to flow when forward biased (p: + ve and n: - ve).\n\nSource: Source 3, Page 18","Page 18, Page 19, Page 18"
5,Explain Boolean algebra briefly.,"Boolean algebra is a branch of Algebra that governs binary digital circuits, where variables can have only two values (ONE or ZERO). It is used to describe different kinds of variables and operations such as OR, AND, and their properties.\n\nSource 1, Page 113; Source 2, Page 107","Page 113, Page 107, Page 114"
6,What is a logic gate?,"A logic gate is a fundamental building block of a digital circuit. It has one or more inputs and usually one output, and the output is determined by the LOGIC followed by that particular gate. \n\nSource: Source 1, Page 107","Page 107, Page 126, Page 107"
7,What is a flip-flop?,"A flip-flop is a basic storage element in sequential circuits that has two outputs, Q and Q, which will always remain in opposite states. It has control inputs that can be used to change (toggle) its present state, allowing it to store one bit of information (1 or 0) indefinitely, as long as the device is powered. (Source 1, Page 126 and Source 2, Page 126)","Page 126, Page 126, Page 131"
8,What is binary coded decimal?,"Binary coded decimal is a method of representing decimal numbers using their binary equivalents.\n\nI used Source 3, Page 106, and Source 2's discussion on binary number systems.","Page 96, Page 96, Page 106"
9,What is a half adder?,"A half adder is a combinational logic circuit that adds two binary numbers, producing a sum (S) and a carry (C) as the output. It has two inputs, called augend and addend bits, and provides two outputs, sum and carry. \n\nSource 2, Page 123","Page 125, Page 123, Page 124"


In [17]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

CUDA available: True
GPU name: Quadro M1200


## 2.5 Vision Component

**Model:** YOLOv8n (nano), fine-tuned on the ElectroCom61 dataset (2121 images, 61 
electronic component classes: resistors, capacitors, transistors, ICs, diodes, etc.)

**Why fine-tune rather than use pretrained as-is:** The pretrained YOLOv8 (COCO weights) 
has no concept of electronic components as object classes, so fine-tuning on ElectroCom61 
is required for meaningful detection.

**Integration with RAG:** When a user uploads an image, YOLO detects and labels the 
electronic component(s) present. The detected class name(s) are injected into the RAG 
prompt as additional context, so the LLM's answer generation is grounded in both the 
textbook content AND what's visually present in the image.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data="../data/images/data.yaml",
    epochs=30,
    imgsz=412,
    batch=4,      
    device=0,       
    project="../data/yolo_runs",
    name="electrocom61",
)

Ultralytics 8.4.146  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (Quadro M1200, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../data/images/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=412, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=electrocom61-4, nbs=64, nms=None, opset=None, optimi

In [ ]:
best_model = YOLO("../data/yolo_runs/electrocom61/weights/best.pt")

test_results = best_model.predict(
    source="../data/images/test/images",
    save=True,
    conf=0.25,
)

print(f"Ran inference on {len(test_results)} test images")
print("Predictions saved to:", test_results[0].save_dir)